In [1]:
import sqlite3
from pathlib import Path

In [2]:
INPUT_DIR = Path(r"C:\Users\alrazz\Documents\Hybrid SP_ID annotation")
OUTPUT_DB = INPUT_DIR / "Combined.db"

# Name of the table in each input database
TABLE_NAME = "texts"

In [3]:
def quote_identifier(name):
    """Safely quote a SQLite table/column name."""
    return '"' + name.replace('"', '""') + '"'


In [4]:
db_files = sorted(INPUT_DIR.glob("*.db"))

# Do not use the output database as an input
db_files = [
    db for db in db_files
    if db.resolve() != OUTPUT_DB.resolve()
]

if not db_files:
    raise FileNotFoundError(
        f"No .db files found in:\n{INPUT_DIR}"
    )

print(f"Found {len(db_files)} input database(s):")
for db in db_files:
    print(f"  - {db.name}")

print()


Found 73 input database(s):
  - ed.db
  - ed_2.db
  - ed_3.db
  - en.db
  - en_10.db
  - en_11.db
  - en_2.db
  - en_3.db
  - en_4.db
  - en_5.db
  - en_6.db
  - en_7.db
  - en_8.db
  - en_9.db
  - fi.db
  - HI_2_clean_dedup.db
  - HI_clean_dedup.db
  - HI_ID_LY_SP_clean2_dedup.db
  - ID_clean_dedup.db
  - it.db
  - it_2.db
  - it_3.db
  - lt.db
  - lt_2.db
  - lt_3.db
  - lt_4.db
  - LY_clean_dedup.db
  - MT.db
  - nb.db
  - nb_2.db
  - nb_3.db
  - nb_4.db
  - nb_5.db
  - nb_6.db
  - nb_7.db
  - New_HI_ID_LY_OP_SP_clean_clean.db
  - ob.db
  - ob_2.db
  - ob_3.db
  - ob_4.db
  - oi.db
  - oi_2.db
  - oi_3.db
  - oo.db
  - oo_2.db
  - oo_3.db
  - oo_4.db
  - oo_5.db
  - oo_6.db
  - os.db
  - os_2.db
  - Persian_data.db
  - Persian_data3_clean.db
  - ra.db
  - ra_2.db
  - ra_3.db
  - ra_4.db
  - re.db
  - re_2.db
  - re_3.db
  - re_4.db
  - rs.db
  - rv.db
  - rv_2.db
  - rv_3.db
  - rv_4.db
  - rv_5.db
  - sample_JNK_7047512_clean.db
  - sample_JNK_7049623_clean.db
  - SP_clean_dedup.db

In [5]:
first_db = db_files[0]

with sqlite3.connect(first_db) as conn:
    cursor = conn.execute(
        f"PRAGMA table_info({quote_identifier(TABLE_NAME)})"
    )
    schema = cursor.fetchall()

if not schema:
    raise RuntimeError(
        f'Table "{TABLE_NAME}" was not found in {first_db.name}'
    )

# PRAGMA table_info returns:
# cid, name, type, notnull, dflt_value, pk

columns = [
    {
        "name": row[1],
        "type": row[2],
        "notnull": row[3],
        "default": row[4],
        "pk": row[5],
    }
    for row in schema
]

column_names = [col["name"] for col in columns]

if "Turku_NLP" not in column_names:
    raise RuntimeError(
        f'The table "{TABLE_NAME}" does not contain a "Turku_NLP" column.'
    )

if "Turku_NLP_sub" not in column_names:
    raise RuntimeError(
        f'The table "{TABLE_NAME}" does not contain a "Turku_NLP_sub" column.'
    )

if "source" in column_names:
    raise RuntimeError(
        'An input database already contains a "source" column. '
        "Please rename/remove it before running this script."
    )


In [6]:
new_columns = []

for col in columns:
    new_columns.append(col)

    # Insert source immediately after Turku_NLP_sub
    if col["name"] == "Turku_NLP_sub":
        new_columns.append({
            "name": "source",
            "type": "TEXT",
            "notnull": 0,
            "default": None,
            "pk": 0,
        })

output_column_names = [col["name"] for col in new_columns]



In [7]:
if OUTPUT_DB.exists():
    print(f"Output database already exists:")
    print(f"  {OUTPUT_DB}")
    print("It will be replaced.\n")

    OUTPUT_DB.unlink()


In [10]:
out_conn = sqlite3.connect(OUTPUT_DB)

try:
    # Build CREATE TABLE statement
    column_definitions = []

    for col in new_columns:
        name = quote_identifier(col["name"])
        col_type = col["type"] or "TEXT"

        column_definitions.append(
            f"{name} {col_type}"
        )

    create_table_sql = f"""
        CREATE TABLE {quote_identifier(TABLE_NAME)} (
            {", ".join(column_definitions)}
        )
    """

    out_conn.execute(create_table_sql)
    out_conn.commit()

    print(f"Created output database:")
    print(f"  {OUTPUT_DB}")
    print()


    # ========================================================
    # PREPARE INSERT STATEMENT
    # ========================================================

    quoted_output_columns = ", ".join(
        quote_identifier(name)
        for name in output_column_names
    )

    placeholders = ", ".join(
        ["?"] * len(output_column_names)
    )

    insert_sql = f"""
        INSERT INTO {quote_identifier(TABLE_NAME)}
        ({quoted_output_columns})
        VALUES ({placeholders})
    """


    # ========================================================
    # PROCESS EACH INPUT DATABASE
    # ========================================================

    total_rows = 0

    for db_file in db_files:

        print(f"Processing: {db_file.name}")

        conn = sqlite3.connect(db_file)

        try:
            # Check that the table exists
            table_exists = conn.execute(
                """
                SELECT name
                FROM sqlite_master
                WHERE type = 'table'
                  AND name = ?
                """,
                (TABLE_NAME,)
            ).fetchone()

            if table_exists is None:
                print(f"  WARNING: No '{TABLE_NAME}' table. Skipping.")
                print()
                continue

            # Check columns in this database
            current_schema = conn.execute(
                f"PRAGMA table_info({quote_identifier(TABLE_NAME)})"
            ).fetchall()

            current_columns = [row[1] for row in current_schema]

            missing_columns = [
                name
                for name in column_names
                if name not in current_columns
            ]

            if missing_columns:
                print(
                    f"  WARNING: Missing columns: {missing_columns}"
                )
                print("  Skipping this database.")
                print()
                continue

            # ------------------------------------------------
            # Build SELECT
            #
            # Turku_NLP must:
            #   - not be NULL
            #   - not be empty
            #   - not contain only whitespace
            # ------------------------------------------------

            select_columns = []

            for name in column_names:

                select_columns.append(
                    quote_identifier(name)
                )

                if name == "Turku_NLP_sub":
                    # source is inserted immediately after
                    # Turku_NLP_sub
                    select_columns.append("? AS source")

            select_sql = f"""
                SELECT {", ".join(select_columns)}
                FROM {quote_identifier(TABLE_NAME)}
                WHERE {quote_identifier("Turku_NLP")} IS NOT NULL
                  AND TRIM({quote_identifier("Turku_NLP")}) <> ''
            """

            # The source filename is used for the ? AS source
            cursor = conn.execute(
                select_sql,
                (db_file.name,)
            )

            rows = cursor.fetchall()

            # Insert into output database
            out_conn.executemany(insert_sql, rows)
            out_conn.commit()

            num_rows = len(rows)
            total_rows += num_rows

            print(f"  Imported: {num_rows:,} rows")

        except sqlite3.Error as e:
            print(f"  ERROR: {e}")
            print("  Skipping this database.")

        finally:
            conn.close()

        print()


    # ========================================================
    # FINALIZE
    # ========================================================

    print("=" * 60)
    print("DONE")
    print("=" * 60)
    print(f"Input databases : {len(db_files):,}")
    print(f"Total rows      : {total_rows:,}")
    print(f"Output database : {OUTPUT_DB}")
    print()


finally:
    out_conn.close()

Created output database:
  C:\Users\alrazz\Documents\Hybrid SP_ID annotation\Combined.db

Processing: ed.db
  Imported: 47 rows

Processing: ed_2.db
  Imported: 98 rows

Processing: ed_3.db
  Imported: 5 rows

Processing: en.db
  Imported: 3 rows

Processing: en_10.db
  Imported: 48 rows

Processing: en_11.db
  Imported: 19 rows

Processing: en_2.db
  Imported: 3 rows

Processing: en_3.db
  Imported: 11 rows

Processing: en_4.db
  Imported: 8 rows

Processing: en_5.db
  Imported: 7 rows

Processing: en_6.db
  Imported: 16 rows

Processing: en_7.db
  Imported: 4 rows

Processing: en_8.db
  Imported: 4 rows

Processing: en_9.db
  Imported: 24 rows

Processing: fi.db
  Imported: 147 rows

Processing: HI_2_clean_dedup.db
  Imported: 85 rows

Processing: HI_clean_dedup.db
  Imported: 61 rows

Processing: HI_ID_LY_SP_clean2_dedup.db
  Imported: 6 rows

Processing: ID_clean_dedup.db
  Imported: 229 rows

Processing: it.db
  Imported: 6 rows

Processing: it_2.db
  Imported: 7 rows

Processing: